# 11 — Recovering a directed interaction network

The model stores `D[target, source]`; a conventional adjacency matrix is `D.T`. Here we plant two disconnected two-chain blocks, generate observations, fit with Dong's linear EM, and ask whether learned mass stays inside the correct block.

In [ ]:
import torch

from latentlinks.influence import InfluenceModel, fit_dong_linear_em
from latentlinks.network import influence_to_adjacency

C = 4
initial = torch.full((C, 2), 0.5)
transition = torch.empty(C, C, 2, 2)
for target in range(C):
    for source in range(C):
        same = target // 2 == source // 2
        transition[target, source] = torch.tensor([[0.9, 0.1], [0.1, 0.9]]) if same else 0.5
emission = torch.tensor([[[0.96, 0.04], [0.04, 0.96]]] * C)
planted = torch.tensor(
    [[0.75, 0.25, 0, 0], [0.30, 0.70, 0, 0], [0, 0, 0.80, 0.20], [0, 0, 0.25, 0.75]]
)
truth = InfluenceModel(initial, transition, emission, planted)
sequences = [truth.sample(120, seed=s)[0] for s in range(8)]
start = InfluenceModel(initial, transition + 0.05, emission + 0.03, torch.full((C, C), 0.25))
fitted, history = fit_dong_linear_em(start, sequences, max_iterations=20)
print(influence_to_adjacency(fitted.influence, include_self=True).round(decimals=2))
inside = torch.stack([fitted.influence[c, (c // 2) * 2 : (c // 2) * 2 + 2].sum() for c in range(C)])
print("within-block mass:", inside.round(decimals=3))
assert torch.all(inside > 0.7)

Recovery needs identifiable dynamics: if every pairwise transition table is identical, changing influence weights changes nothing observable. Even when identifiable, these edges mean predictive dynamical dependence—not intervention-based causality.